# 015 Models Playaround\n\n這份 notebook 是一條可分段執行的 pipeline：\n1) dataset generation（依照 `feature_count` 產生 Z-score features + sequences）\n2) TCN training（`num_inputs` 會自動跟著 feature_count 變動）\n3) basic evaluation（先輸出 JSON 指標；完整圖表/CSV 之後再加）\n\n所有輸出都會寫到 `new_era/15_models_playaround/015_models_playaround_result/<experiment_id>/...`。

In [ ]:
from new_era.models_playaround_15.config import (\n    BacktestConfig,\n    DataConfig,\n    ExperimentConfig,\n    FeatureSet,\n    LabelConfig,\n    ModelConfig,\n    SplitConfig,\n    TrainingConfig,\n    WindowConfig,\n)\nfrom new_era.models_playaround_15.paths import build_experiment_paths, experiment_id, get_project_root, get_results_root\n\nproject_root = get_project_root()\n\ncfg = ExperimentConfig(\n    data=DataConfig(\n        data_dir=project_root / "data" / "NEW",\n        symbols=("TXFB6",),\n        zscore_window=2000,\n    ),\n    window=WindowConfig(\n        window_size=100,\n        horizon=10,\n        threshold_ticks=1.0,\n        sample_stride_train=5,\n    ),\n    split=SplitConfig(train_session_frac=0.70, val_session_frac=0.15),\n    features=FeatureSet(feature_count=8),\n    labels=LabelConfig(label_type="mid_move_3class"),\n    model=ModelConfig(channel_list=(16, 16, 16), kernel_size=3, dropout=0.1),\n    train=TrainingConfig(epochs=20, batch_size=1024, lr=1e-3, seed=42, device="auto"),\n    backtest=BacktestConfig(symbol="TXFB6"),\n)\n\nprint("experiment_id:", experiment_id(cfg))\nprint("num_inputs:", cfg.num_inputs, "num_classes:", cfg.num_classes)\nprint("results_root:", get_results_root())\n\npaths = build_experiment_paths(cfg)\npaths.root_dir

In [ ]:
from new_era.models_playaround_15.dataset_generation import generate_dataset_npz\n\nmanifest = generate_dataset_npz(cfg, paths)\nlen(manifest["sessions"]), list(manifest["class_distribution_by_session"].items())[:2]

In [ ]:
from new_era.models_playaround_15.train_tcn import train_tcn\n\ntrain_summary = train_tcn(cfg, paths)\ntrain_summary

In [ ]:
from new_era.models_playaround_15.evaluate_tcn_full import evaluate_full\n\nsummary = evaluate_full(cfg, paths)\nsummary["splits"]["val"]["accuracy"]

## 下一步\n\n- `feature_count` ablation：把 `FeatureSet(feature_count=6/7/8/9)` 跑一輪。\n- label types：接下來會把新的 label generation 接到同一條 pipeline 裡（先確保 `mid_move_3class` baseline 不壞）。\n- evaluation：依你的規格輸出 confusion matrix / report CSV / PR-ROC-calibration / threshold diagnostics。\n- backtest：改成 validation sweep 選 threshold，test 固定一次。

In [ ]:
# Backtest: val sweep -> fixed test (maker-only)\nfrom new_era.models_playaround_15.backtest_val_sweep import backtest_val_sweep_then_test\n\nbt_out = backtest_val_sweep_then_test(cfg, paths)\nbt_out["test_final"]